In [0]:
import requests

url = "https://api.exchangerate-api.com/v4/latest/USD"

try:
    response = requests.get(url, timeout=10)
    print("Status code:", response.status_code)
    print("Resposta:", response.json())
except Exception as e:
    print("ERRO ao conectar:", e)

In [0]:
%pip install faker

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
sys.path.append("/Workspace/Users/bruno.quelestech@outlook.com/poc-lakehouse-food-latam/src")

from utils.faker_helper import FakerHelper

fh = FakerHelper(pais="brasil")
print("Nome:", fh.gerar_nome())
print("Cidade:", fh.gerar_cidade())
print("Telefone:", fh.gerar_telefone())

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
sys.path.append("/Workspace/Users/bruno.quelestech@outlook.com/poc-lakehouse-food-latam/src")

from utils.faker_helper import FakerHelper
from utils.scd2_handler import SCD2Handler

print("Importações OK")

In [0]:
from utils.scd2_handler import SCD2Handler

# Criando um DataFrame de exemplo, simulando 2 produtos "crus"
# (sem nenhuma coluna de controle SCD2 ainda)
dados_exemplo = [
    ("PROD001", "MAIONESE", "Maionese", "Mayonesa", "Mayonesa", "Mayonnaise", "1kg"),
    ("PROD002", "MOSTARDA", "Mostarda", "Mostaza", "Mostaza", "Mustard", "5kg"),
]

colunas = [
    "produto_id", "nome_interno", "nome_brasil",
    "nome_argentina", "nome_mexico", "nome_ingles", "tamanho"
]

df_produtos_cru = spark.createDataFrame(dados_exemplo, colunas)

# Aplicando o controle SCD2
scd2 = SCD2Handler()
df_produtos_com_scd2 = scd2.iniciar_controle_scd2(df_produtos_cru)

df_produtos_com_scd2.display()

In [0]:
# Verificação de cobertura de dias na Silver

spark.table("poc_latam_food.silver.fato_vendas") \
    .select("data_venda") \
    .distinct() \
    .orderBy("data_venda") \
    .display()

In [0]:
# Verificação final - Gold também está atualizada

spark.table("poc_latam_food.gold.sales_global").orderBy("period").display()

In [0]:
# Estado atual da Raw - partições e contagens

spark.table("poc_latam_food.raw.vendas") \
    .groupBy("data_ingestao_particao") \
    .count() \
    .orderBy("data_ingestao_particao") \
    .display()

print(f"Total atual na Raw: {spark.table('poc_latam_food.raw.vendas').count()}")

In [0]:
from datetime import datetime, timedelta

data_limite = (datetime.now() - timedelta(hours=48)).date()
print(f"Data-limite (48h atrás): {data_limite}")
print(f"Partições com data_ingestao_particao < '{data_limite}' seriam removidas.")

In [0]:
spark.table("poc_latam_food.raw.vendas") \
    .groupBy("data_ingestao_particao") \
    .count() \
    .orderBy("data_ingestao_particao") \
    .display()

In [0]:
# Contagem por país - partição de hoje

spark.table("poc_latam_food.raw.vendas") \
    .filter("data_ingestao_particao = '2026-07-27'") \
    .groupBy("pais") \
    .count() \
    .orderBy("pais") \
    .display()

In [0]:
# Verificação de duplicidade de venda_id - Argentina, hoje

spark.table("poc_latam_food.raw.vendas") \
    .filter("data_ingestao_particao = '2026-07-27' AND pais = 'argentina'") \
    .groupBy("venda_id") \
    .count() \
    .filter("count > 1") \
    .display()

In [0]:
# Verificação de arquivos na Landing Zone - Argentina, hoje

display(dbutils.fs.ls("/Volumes/poc_latam_food/landing/blob_simulado/vendas/pais=argentina/data=2026-07-27/"))

In [0]:
# Investigação - horários de ingestão das vendas da Argentina, hoje

spark.table("poc_latam_food.raw.vendas") \
    .filter("data_ingestao_particao = '2026-07-27' AND pais = 'argentina'") \
    .groupBy("data_ingestao") \
    .count() \
    .orderBy("data_ingestao") \
    .display()

In [0]:
# Identificação de 500 vendas a remover (arbitrário, mantendo consistência)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_argentina_duplicado = spark.table("poc_latam_food.raw.vendas") \
    .filter("data_ingestao_particao = '2026-07-27' AND pais = 'argentina'")

window_spec = Window.orderBy("venda_id")

df_com_row_number = df_argentina_duplicado.withColumn("rn", row_number().over(window_spec))

ids_para_remover = [row["venda_id"] for row in df_com_row_number.filter("rn > 500").select("venda_id").collect()]

print(f"Total de IDs identificados para remoção: {len(ids_para_remover)}")

In [0]:
# Criando view temporária com os IDs identificados

df_ids_remover = spark.createDataFrame([(vid,) for vid in ids_para_remover], ["venda_id"])
df_ids_remover.createOrReplaceTempView("ids_para_remover")

print(f"View temporária criada com {df_ids_remover.count()} IDs.")

In [0]:
# DELETE na Raw

spark.sql("""
    DELETE FROM poc_latam_food.raw.vendas
    WHERE venda_id IN (SELECT venda_id FROM ids_para_remover)
""")

print(f"Total na Raw após remoção: {spark.table('poc_latam_food.raw.vendas').count()}")

In [0]:
# DELETE na Bronze

spark.sql("""
    DELETE FROM poc_latam_food.bronze.vendas
    WHERE venda_id IN (SELECT venda_id FROM ids_para_remover)
""")

print(f"Total na Bronze após remoção: {spark.table('poc_latam_food.bronze.vendas').count()}")

In [0]:
# Verificação - contagem por país, Bronze, hoje

spark.table("poc_latam_food.bronze.vendas") \
    .filter("data_ingestao_particao = '2026-07-27'") \
    .groupBy("pais") \
    .count() \
    .orderBy("pais") \
    .display()

In [0]:
# Verificação do total atual da Silver (antes de decidir se precisa deletar)

print(f"Total atual na Silver: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

spark.table("poc_latam_food.silver.fato_vendas") \
    .filter("data_venda = '2026-07-27'") \
    .groupBy("pais") \
    .count() \
    .orderBy("pais") \
    .display()

In [0]:
# DELETE na Silver

spark.sql("""
    DELETE FROM poc_latam_food.silver.fato_vendas
    WHERE venda_id IN (SELECT venda_id FROM ids_para_remover)
""")

print(f"Total na Silver após remoção: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

In [0]:
# Validação final - totais em todas as camadas

print(f"Raw: {spark.table('poc_latam_food.raw.vendas').count()}")
print(f"Bronze: {spark.table('poc_latam_food.bronze.vendas').count()}")
print(f"Silver: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

print("\nGold - sales_global:")
spark.table("poc_latam_food.gold.sales_global").orderBy("period").display()

print("\nGold - sales_by_country (Argentina, todos os dias):")
spark.table("poc_latam_food.gold.sales_by_country") \
    .filter("country = 'Argentina'") \
    .orderBy("period") \
    .display()

In [0]:
# Verificação consolidada - todas as camadas

print("=== RAW ===")
spark.table("poc_latam_food.raw.vendas").groupBy("data_ingestao_particao").count().orderBy("data_ingestao_particao").show()

print("=== BRONZE (contagem total e por país/dia) ===")
print(f"Total: {spark.table('poc_latam_food.bronze.vendas').count()}")
spark.table("poc_latam_food.bronze.vendas").groupBy("data_venda", "pais").count().orderBy("data_venda", "pais").show(50)

print("=== SILVER (contagem total e por país/dia) ===")
print(f"Total: {spark.table('poc_latam_food.silver.fato_vendas').count()}")
spark.table("poc_latam_food.silver.fato_vendas").groupBy("data_venda", "pais").count().orderBy("data_venda", "pais").show(50)

print("=== GOLD - sales_global ===")
spark.table("poc_latam_food.gold.sales_global").orderBy("period").show()

print("=== GOLD - sales_by_country (conferindo Argentina especificamente) ===")
spark.table("poc_latam_food.gold.sales_by_country").filter("country = 'Argentina'").orderBy("period").show()

In [0]:
# Investigação - venda_id duplicados na Bronze, dia 27/07

spark.table("poc_latam_food.bronze.vendas") \
    .filter("data_venda = '2026-07-27'") \
    .groupBy("venda_id") \
    .count() \
    .filter("count > 1") \
    .display()

print("Total de venda_id distintos com duplicidade na Bronze (27/07):")
print(
    spark.table("poc_latam_food.bronze.vendas")
    .filter("data_venda = '2026-07-27'")
    .groupBy("venda_id")
    .count()
    .filter("count > 1")
    .count()
)

In [0]:
# Investigação - venda_id duplicados na Silver, dia 27/07

spark.table("poc_latam_food.silver.fato_vendas") \
    .filter("data_venda = '2026-07-27'") \
    .groupBy("venda_id") \
    .count() \
    .groupBy("count") \
    .count() \
    .orderBy("count") \
    .show()

In [0]:
spark.sql("""
    DELETE FROM poc_latam_food.silver.fato_vendas
    WHERE data_venda = '2026-07-27'
""")

print(f"Total na Silver após remoção: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

In [0]:
spark.sql("""
    DELETE FROM poc_latam_food.bronze.vendas
    WHERE data_venda = '2026-07-27'
""")

print(f"Total na Bronze após remoção: {spark.table('poc_latam_food.bronze.vendas').count()}")

In [0]:
# Verificação do histórico da tabela raw.vendas ao redor do dia 27/07

spark.sql("DESCRIBE HISTORY poc_latam_food.raw.vendas") \
    .select("version", "timestamp", "operation", "operationMetrics") \
    .orderBy("version") \
    .show(50, truncate=False)

In [0]:
# Backfill - leitura batch do dia 27/07 da Raw

df_raw_27 = spark.read.table("poc_latam_food.raw.vendas").filter("data_venda = '2026-07-27'")

print(f"Total de linhas lidas da Raw para 27/07: {df_raw_27.count()}")
df_raw_27.groupBy("pais").count().orderBy("pais").show()

In [0]:
# Backfill - gravação em Bronze (batch, sem streaming)

from pyspark.sql.functions import current_timestamp

df_bronze_27 = df_raw_27.withColumn("data_ingestao_bronze", current_timestamp())

df_bronze_27.write.format("delta").mode("append").saveAsTable("poc_latam_food.bronze.vendas")

print(f"Total na Bronze após backfill: {spark.table('poc_latam_food.bronze.vendas').count()}")

In [0]:
# Backfill - recriando a lógica da Silver para 27/07 (batch)

from pyspark.sql.functions import col, regexp_replace, upper, translate, when

caracteres_com_acento = "áàãâäéèêëíìîïóòõôöúùûüçñÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇÑ"
caracteres_sem_acento = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"

df_vendas_tratado_27 = (
    df_bronze_27
    .withColumn(
        "valor_unitario_moeda_local",
        regexp_replace(col("valor_unitario_moeda_local"), ",", ".").cast("decimal(10,2)")
    )
    .withColumn("valor_total_moeda_local", col("quantidade") * col("valor_unitario_moeda_local"))
    .withColumn("loja_cidade", upper(translate(col("loja_cidade"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn("pais", upper(translate(col("pais"), caracteres_com_acento, caracteres_sem_acento)))
)

print(f"Total após tratamento: {df_vendas_tratado_27.count()}")
df_vendas_tratado_27.select("valor_unitario_moeda_local", "valor_total_moeda_local", "loja_cidade", "pais").show(5)

In [0]:
# Backfill - JOINs com dimensões e câmbio para 27/07

df_lojas_silver = spark.table("poc_latam_food.silver.lojas")
df_representantes_silver = spark.table("poc_latam_food.silver.representantes")
df_cambio_silver = spark.table("poc_latam_food.silver.dim_cambio")

# JOIN com Lojas (efetivo por data)
df_com_loja_27 = (
    df_vendas_tratado_27.alias("v")
    .join(
        df_lojas_silver.alias("l"),
        (col("v.loja_cidade") == col("l.nome_cidade")) &
        (col("v.data_venda").cast("date") >= col("l.data_inicio")) &
        (col("l.data_fim").isNull() | (col("v.data_venda").cast("date") < col("l.data_fim"))),
        "left"
    )
    .select("v.*", col("l.loja_id").alias("loja_id"))
)

# JOIN com Representantes (efetivo por data)
df_com_representante_27 = (
    df_com_loja_27.alias("v")
    .join(
        df_representantes_silver.alias("r"),
        (col("v.representante_id") == col("r.representante_id")) &
        (col("v.data_venda").cast("date") >= col("r.data_inicio")) &
        (col("r.data_fim").isNull() | (col("v.data_venda").cast("date") < col("r.data_fim"))),
        "left"
    )
    .select("v.*", col("r.nome").alias("representante_nome"))
)

# JOIN com Câmbio
from pyspark.sql.functions import round as spark_round

df_com_cambio_27 = (
    df_com_representante_27.alias("v")
    .join(
        df_cambio_silver.alias("c"),
        (col("v.moeda") == col("c.moeda")) &
        (col("v.data_venda").cast("date") == col("c.data_cotacao")),
        "left"
    )
    .select(
        "v.*",
        col("c.taxa_para_usd").alias("cambio_usado"),
        spark_round(col("v.valor_total_moeda_local") / col("c.taxa_para_usd"), 2).alias("valor_total_usd")
    )
)

print(f"Total após JOINs: {df_com_cambio_27.count()}")
df_com_cambio_27.select("venda_id", "loja_id", "representante_nome", "cambio_usado", "valor_total_usd").show(5)

In [0]:
# Backfill - seleção final e gravação na Silver

from pyspark.sql.functions import current_timestamp

df_silver_final_27 = (
    df_com_cambio_27
    .select(
        "venda_id",
        col("data_venda").cast("date").alias("data_venda"),
        "produto_id",
        "loja_id",
        "loja_cidade",
        "representante_id",
        "representante_nome",
        "pais",
        "quantidade",
        "valor_unitario_moeda_local",
        "valor_total_moeda_local",
        "moeda",
        "cambio_usado",
        "valor_total_usd",
        "data_ingestao",
        "data_ingestao_bronze",
        current_timestamp().alias("data_ingestao_silver")
    )
)

df_silver_final_27.write.format("delta").mode("append").saveAsTable("poc_latam_food.silver.fato_vendas")

print(f"Total na Silver após backfill: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

In [0]:
# Validação final pós-backfill

print(f"Bronze: {spark.table('poc_latam_food.bronze.vendas').count()}")
print(f"Silver: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

print("\nSilver por país/dia (27/07):")
spark.table("poc_latam_food.silver.fato_vendas").filter("data_venda = '2026-07-27'").groupBy("pais").count().orderBy("pais").show()

print("Confirmando ausência de duplicatas (27/07):")
spark.table("poc_latam_food.silver.fato_vendas").filter("data_venda = '2026-07-27'").groupBy("venda_id").count().filter("count > 1").count()

In [0]:
spark.table("poc_latam_food.silver.dim_cambio").orderBy("data_cotacao").show()

In [0]:
spark.table("poc_latam_food.silver.fato_vendas").filter("data_venda = '2026-07-28'").groupBy("pais").count().orderBy("pais").show()

print("Verificando valor_total_usd para o dia 28:")
spark.table("poc_latam_food.silver.fato_vendas").filter("data_venda = '2026-07-28'").select("valor_total_usd").distinct().show()

In [0]:
# Backfill de câmbio - dia 28/07 (calculando valores corretos)

from pyspark.sql.functions import col, round as spark_round

df_cambio_28 = spark.table("poc_latam_food.silver.dim_cambio").filter("data_cotacao = '2026-07-28'")

df_vendas_28_corrigido = (
    spark.table("poc_latam_food.silver.fato_vendas")
    .filter("data_venda = '2026-07-28'")
    .alias("v")
    .join(
        df_cambio_28.alias("c"),
        col("v.moeda") == col("c.moeda"),
        "left"
    )
    .select(
        col("v.venda_id"),
        col("c.taxa_para_usd").alias("cambio_usado_novo"),
        spark_round(col("v.valor_total_moeda_local") / col("c.taxa_para_usd"), 2).alias("valor_total_usd_novo")
    )
)

print(f"Total de linhas a corrigir: {df_vendas_28_corrigido.count()}")
df_vendas_28_corrigido.show(5)

In [0]:
# Backfill de câmbio - MERGE aplicando a correção

from delta.tables import DeltaTable

tabela_silver_vendas = DeltaTable.forName(spark, "poc_latam_food.silver.fato_vendas")

(
    tabela_silver_vendas.alias("target")
    .merge(
        df_vendas_28_corrigido.alias("source"),
        "target.venda_id = source.venda_id"
    )
    .whenMatchedUpdate(set={
        "cambio_usado": col("source.cambio_usado_novo"),
        "valor_total_usd": col("source.valor_total_usd_novo")
    })
    .execute()
)

print("MERGE concluído.")

In [0]:
print("Verificando valor_total_usd para o dia 28 após MERGE:")
spark.table("poc_latam_food.silver.fato_vendas").filter("data_venda = '2026-07-28'").select("valor_total_usd").filter("valor_total_usd IS NULL").count()

print(f"\nTotal na Silver: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

In [0]:
total_nulos_28 = spark.table("poc_latam_food.silver.fato_vendas").filter("data_venda = '2026-07-28'").filter("valor_total_usd IS NULL").count()
print(f"Total de nulos no dia 28 após MERGE: {total_nulos_28}")

In [0]:
# Validação final - todas as camadas

print("=== BRONZE ===")
print(f"Total: {spark.table('poc_latam_food.bronze.vendas').count()}")

print("\n=== SILVER ===")
print(f"Total: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

print("\nSilver - verificação de duplicatas globais:")
duplicatas = (
    spark.table("poc_latam_food.silver.fato_vendas")
    .groupBy("venda_id")
    .count()
    .filter("count > 1")
    .count()
)
print(f"Total de venda_id duplicados: {duplicatas}")

print("\n=== GOLD - sales_global ===")
spark.table("poc_latam_food.gold.sales_global").orderBy("period").show()

print("=== GOLD - sales_by_country (Argentina) ===")
spark.table("poc_latam_food.gold.sales_by_country").filter("country = 'Argentina'").orderBy("period").show()

In [0]:
# Validação pós-Run now

print("=== BRONZE ===")
print(f"Total: {spark.table('poc_latam_food.bronze.vendas').count()}")

print("\n=== SILVER ===")
print(f"Total: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

print("\nVerificação de duplicatas globais:")
duplicatas = (
    spark.table("poc_latam_food.silver.fato_vendas")
    .groupBy("venda_id")
    .count()
    .filter("count > 1")
    .count()
)
print(f"Total de venda_id duplicados: {duplicatas}")

print("\n=== RAW ===")
spark.table("poc_latam_food.raw.vendas").groupBy("data_ingestao_particao").count().orderBy("data_ingestao_particao").show()

print("=== GOLD - sales_global ===")
spark.table("poc_latam_food.gold.sales_global").orderBy("period").show()

In [0]:
# Correção - remover duplicata do dia 27 na Silver, mantendo 1 cópia por venda_id

df_dia27_dedup = (
    spark.table("poc_latam_food.silver.fato_vendas")
    .filter("data_venda = '2026-07-27'")
    .dropDuplicates(["venda_id"])
)

print(f"Total após dedup: {df_dia27_dedup.count()}")

spark.sql("DELETE FROM poc_latam_food.silver.fato_vendas WHERE data_venda = '2026-07-27'")

df_dia27_dedup.write.format("delta").mode("append").saveAsTable("poc_latam_food.silver.fato_vendas")

print(f"Total na Silver após correção: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

In [0]:
# Correção definitiva - materializando antes do DELETE

df_dia27_dedup = (
    spark.table("poc_latam_food.silver.fato_vendas")
    .filter("data_venda = '2026-07-27'")
    .dropDuplicates(["venda_id"])
)

# Grava em uma tabela temporária ANTES de qualquer DELETE, evitando o problema de lazy evaluation
df_dia27_dedup.write.format("delta").mode("overwrite").saveAsTable("poc_latam_food.silver._staging_dia27")

print(f"Total salvo no staging: {spark.table('poc_latam_food.silver._staging_dia27').count()}")

In [0]:
# Backfill (retry) - lendo o dia 27/07 direto da Bronze

df_bronze_27_retry = spark.read.table("poc_latam_food.bronze.vendas").filter("data_venda = '2026-07-27'")

print(f"Total lido da Bronze para 27/07: {df_bronze_27_retry.count()}")

In [0]:
# Backfill (retry) - transformação completa

from pyspark.sql.functions import col, regexp_replace, upper, translate, round as spark_round, current_timestamp

caracteres_com_acento = "áàãâäéèêëíìîïóòõôöúùûüçñÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇÑ"
caracteres_sem_acento = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"

df_tratado = (
    df_bronze_27_retry
    .withColumn("valor_unitario_moeda_local", regexp_replace(col("valor_unitario_moeda_local"), ",", ".").cast("decimal(10,2)"))
    .withColumn("valor_total_moeda_local", col("quantidade") * col("valor_unitario_moeda_local"))
    .withColumn("loja_cidade", upper(translate(col("loja_cidade"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn("pais", upper(translate(col("pais"), caracteres_com_acento, caracteres_sem_acento)))
)

df_lojas_silver = spark.table("poc_latam_food.silver.lojas")
df_representantes_silver = spark.table("poc_latam_food.silver.representantes")
df_cambio_silver = spark.table("poc_latam_food.silver.dim_cambio")

df_com_loja = (
    df_tratado.alias("v")
    .join(df_lojas_silver.alias("l"),
          (col("v.loja_cidade") == col("l.nome_cidade")) &
          (col("v.data_venda").cast("date") >= col("l.data_inicio")) &
          (col("l.data_fim").isNull() | (col("v.data_venda").cast("date") < col("l.data_fim"))), "left")
    .select("v.*", col("l.loja_id").alias("loja_id"))
)

df_com_representante = (
    df_com_loja.alias("v")
    .join(df_representantes_silver.alias("r"),
          (col("v.representante_id") == col("r.representante_id")) &
          (col("v.data_venda").cast("date") >= col("r.data_inicio")) &
          (col("r.data_fim").isNull() | (col("v.data_venda").cast("date") < col("r.data_fim"))), "left")
    .select("v.*", col("r.nome").alias("representante_nome"))
)

df_com_cambio = (
    df_com_representante.alias("v")
    .join(df_cambio_silver.alias("c"),
          (col("v.moeda") == col("c.moeda")) &
          (col("v.data_venda").cast("date") == col("c.data_cotacao")), "left")
    .select("v.*",
            col("c.taxa_para_usd").alias("cambio_usado"),
            spark_round(col("v.valor_total_moeda_local") / col("c.taxa_para_usd"), 2).alias("valor_total_usd"))
)

df_final_27 = df_com_cambio.select(
    "venda_id", col("data_venda").cast("date").alias("data_venda"), "produto_id", "loja_id", "loja_cidade",
    "representante_id", "representante_nome", "pais", "quantidade", "valor_unitario_moeda_local",
    "valor_total_moeda_local", "moeda", "cambio_usado", "valor_total_usd",
    "data_ingestao", "data_ingestao_bronze", current_timestamp().alias("data_ingestao_silver")
)

print(f"Total transformado: {df_final_27.count()}")

In [0]:
# Materializando o resultado ANTES de qualquer DELETE

df_final_27.write.format("delta").mode("overwrite").saveAsTable("poc_latam_food.silver._staging_dia27")

print(f"Total salvo no staging: {spark.table('poc_latam_food.silver._staging_dia27').count()}")

In [0]:
# Inserindo os dados do staging na Silver

df_staging = spark.table("poc_latam_food.silver._staging_dia27")

df_staging.write.format("delta").mode("append").saveAsTable("poc_latam_food.silver.fato_vendas")

print(f"Total na Silver após reinserção: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

In [0]:
print(f"Silver total: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

duplicatas = spark.table("poc_latam_food.silver.fato_vendas").groupBy("venda_id").count().filter("count > 1").count()
print(f"Duplicatas: {duplicatas}")

spark.table("poc_latam_food.silver.fato_vendas").filter("data_venda = '2026-07-27'").groupBy("pais").count().orderBy("pais").show()

In [0]:
spark.sql("DROP TABLE IF EXISTS poc_latam_food.silver._staging_dia27")
print("Tabela de staging removida.")

In [0]:
print("=== BRONZE ===")
print(f"Total: {spark.table('poc_latam_food.bronze.vendas').count()}")

print("\n=== SILVER ===")
print(f"Total: {spark.table('poc_latam_food.silver.fato_vendas').count()}")

duplicatas = spark.table("poc_latam_food.silver.fato_vendas").groupBy("venda_id").count().filter("count > 1").count()
print(f"Duplicatas: {duplicatas}")

print("\n=== GOLD - sales_global ===")
spark.table("poc_latam_food.gold.sales_global").orderBy("period").show()